## Chaper 2: Working with Text Data

### 2.2 Tokenizing Text

In [1]:
import os
import requests

filepath = "data/the-verdict.txt"

if not os.path.exists(filepath):
    url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/refs/heads/main/ch02/01_main-chapter-code/the-verdict.txt"

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"
    }

    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()

    with open(filepath, "w", encoding="utf-8") as f:
        f.write(response.text)

    print("Download successful!")

In [2]:
with open(filepath, "r", encoding="utf-8") as f:
    raw_text = f.read()

len(raw_text)

20479

In [3]:
import re

text = "Hello, world. This, is a test."
result = re.split(r'([,.]\s)', text)

result = [item for item in result if item.strip()]

result

['Hello', ', ', 'world', '. ', 'This', ', ', 'is a test.']

In [4]:
text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)

preprocessed = [item for item in result if item.strip()]

len(preprocessed)

4690

In [5]:
preprocessed[:10]

['I',
 'HAD',
 'always',
 'thought',
 'Jack',
 'Gisburn',
 'rather',
 'a',
 'cheap',
 'genius']

### 2.3 Converting tokens into token IDs

In [6]:
all_words = sorted(set(preprocessed))

vocab_size = len(all_words)

vocab_size

1130

In [7]:
vocab = {token:integer for integer, token in enumerate(all_words)}

In [45]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
        
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        
        preprocessed = [item for item in preprocessed if item.strip()]
        
        ids = [self.str_to_int[s] for s in preprocessed]
        
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace punctuation before adding white spce
        text = re.sub(r'\s+([,.?!"()\]])', r'\1', text)
        return text

In [35]:
tokenizer = SimpleTokenizerV1(vocab)


In [36]:
text = """"It's the last he painted, you know,"
        Mrs. Gisburn said with pardonable pride."""

In [37]:
ids = tokenizer.encode(text)
ids

[1,
 56,
 2,
 850,
 988,
 602,
 533,
 746,
 5,
 1126,
 596,
 5,
 1,
 67,
 7,
 38,
 851,
 1108,
 754,
 793,
 7]

In [38]:
tokenizer.decode(ids)

'" It \' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

### 2.3 Adding special context tokens

In [39]:
text = "Hello, do you like tea. is this-- a test?"

In [40]:
tokenizer.encode(text)

KeyError: 'Hello'

In [42]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer, token in enumerate(all_tokens)}

len(vocab)

1132

In [47]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
        
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        
        preprocessed = [item for item in preprocessed if item.strip()]
        
        preprocessed = [item if item in self.str_to_int else "<|unk|>" for item in preprocessed]
        
        ids = [self.str_to_int[s] for s in preprocessed]
        
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace punctuation before adding white spce
        text = re.sub(r'\s+([,.?!"()\]])', r'\1', text)
        return text

In [48]:
tokenizer = SimpleTokenizerV2(vocab)


In [49]:
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 7, 584, 999, 6, 115, 1131, 10]

In [50]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea. is this -- a <|unk|>?'

### 2.5 Byte pair encoding

https://github.com/rasbt/LLMs-from-scratch/tree/main/ch02/05_bpe-from-scratch

In [52]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
tokenizer

<Encoding 'gpt2'>

In [53]:
tokenizer.encode("Hello World!")

[15496, 2159, 0]

In [54]:
tokenizer.decode(tokenizer.encode("Hello World!"))

'Hello World!'

In [57]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
    "of someunknownPlace."
)

tokenizer.encode(text, allowed_special={"<|endoftext|>"})

[15496,
 11,
 466,
 345,
 588,
 8887,
 30,
 220,
 50256,
 554,
 262,
 4252,
 18250,
 8812,
 2114,
 1659,
 617,
 34680,
 27271,
 13]

### 2.6 Data Sampling with sliding window

In [58]:
enc_text = tokenizer.encode(raw_text)
len(enc_text)

5145

In [61]:
enc_sample = enc_text[50:]
len(enc_sample)

5095

In [60]:
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


In [64]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    
    print(tokenizer.decode(context), "----->", tokenizer.decode([desired]))

 and ----->  established
 and established ----->  himself
 and established himself ----->  in
 and established himself in ----->  a


In [65]:
import torch

In [71]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Modification
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        # token_ids = [int(i) for i in txt.strip().split()]

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [69]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [75]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]]), tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])]


In [76]:
for inputs, targets in dataloader:
    pass

print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[  290,  8104,   465,  1021],
        [  319,   616,  8163,   351],
        [  257,  6487,    13,   366],
        [10049,   262, 21296,   286],
        [  340,   318,   326,   314],
        [ 4808,   321,    62,   991],
        [12036,   438, 20777, 41379],
        [  293,   338,  1804,   340]])

Targets:
 tensor([[ 8104,   465,  1021,   319],
        [  616,  8163,   351,   257],
        [ 6487,    13,   366, 10049],
        [  262, 21296,   286,   340],
        [  318,   326,   314,  4808],
        [  321,    62,   991, 12036],
        [  438, 20777, 41379,   293],
        [  338,  1804,   340,   329]])


### 2.7 Creating token embeddings

In [82]:
torch.manual_seed(123)

inputs_ids = torch.tensor([2, 3, 5, 1])

vocab_size = 6
output_dim = 3

embedding_layer = torch.nn.Embedding(tokenizer.n_vocab, output_dim)

embedding_layer.weight

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.3035],
        [-0.5880,  0.3486,  0.6603],
        [-0.2196, -0.3792,  0.7671],
        ...,
        [-0.5931,  1.0895, -0.6854],
        [ 0.7447,  0.5803, -0.4246],
        [-0.3130,  0.7558, -1.2656]], requires_grad=True)

In [83]:
embedding_layer(torch.tensor([3]))

tensor([[-1.1925,  0.6984, -1.4097]], grad_fn=<EmbeddingBackward0>)

In [84]:
embedding_layer(inputs_ids)

tensor([[-0.2196, -0.3792,  0.7671],
        [-1.1925,  0.6984, -1.4097],
        [ 0.2692, -0.0770, -1.0205],
        [-0.5880,  0.3486,  0.6603]], grad_fn=<EmbeddingBackward0>)

### 2.8 Encoding words positions

In [85]:
vocab_size = tokenizer.n_vocab
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

In [88]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)

print("Inputs:\n", inputs)
print("\Inputs shape:\n", inputs.shape)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
\Inputs shape:
 torch.Size([8, 4])


In [89]:
token_embeddings = token_embedding_layer(inputs)
token_embeddings.shape

torch.Size([8, 4, 256])

In [90]:
context_length = 4
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

In [91]:
torch.arange(4)

tensor([0, 1, 2, 3])

In [92]:
pos_embedding_layer.weight

Parameter containing:
tensor([[-0.3974,  0.9388,  1.4840,  ..., -0.2571,  0.9362,  0.0156],
        [-3.0412, -1.4655,  1.0127,  ..., -0.3643,  1.4198, -1.9203],
        [-0.9564,  0.2047, -0.5263,  ..., -0.4493, -1.0943, -0.9001],
        [-1.0230,  0.1866, -0.6785,  ..., -1.1320, -1.0683,  1.3316]],
       requires_grad=True)

In [93]:
pos_embeddings = pos_embedding_layer(torch.arange(4))
pos_embeddings

tensor([[-0.3974,  0.9388,  1.4840,  ..., -0.2571,  0.9362,  0.0156],
        [-3.0412, -1.4655,  1.0127,  ..., -0.3643,  1.4198, -1.9203],
        [-0.9564,  0.2047, -0.5263,  ..., -0.4493, -1.0943, -0.9001],
        [-1.0230,  0.1866, -0.6785,  ..., -1.1320, -1.0683,  1.3316]],
       grad_fn=<EmbeddingBackward0>)

In [94]:
input_embeddings = token_embeddings + pos_embeddings

input_embeddings.shape

torch.Size([8, 4, 256])